# OSCAR Baseline Model

This architecture is a simpler light scaled  OSCAR-style baseline which uses Nico++ unique dataset. To ensure the model trains in a realistic time frame we made slight adjustments as highlighted below. Comparative analysis will be made with this baseline after our top-k vectors compression is performed. Further refinements will be made to the model such as regularization, and scheduled learning rate before final evaluation is done in next deliverable.

Pipeline: **real images → BLIP captions → CLIP text embeddings → optional Stable Diffusion synthesis → ResNet-18 classifier**

### Why this version
- Uses **NICO++ only**, actual model can be trained on nico++ common, domainnet and open images datasets.
- Defaults to a small number of classes and images so it can run in a realistic timeframe.
- Uses ResNet-18, compared to better than ResNet-50 or 101.

## 1) Setup


In [ ]:
!pip -q install transformers diffusers accelerate safetensors

!pip install torch torchvision torchaudio
# !pip install torch==2.1.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")


import os
import json
import math
import random
import shutil
import warnings
from collections import defaultdict, Counter
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



warnings.filterwarnings("ignore")

In [ ]:
SEED = 28
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print("Device:", DEVICE)
print("Torch:", torch.__version__)

Device: cuda
Torch: 2.10.0+cu128


## 2) Configuration

This notebook uses **NICO++ only** and selects the **top 15 classes by image count**.

In [ ]:
@dataclass
class Config:
    dataset_name: str = "nico_unique"
    data_root: str = "/kaggle/input/datasets/khizaryy/nicoplusplus/NICO_unique"

    # Data limits
    max_classes: int = 30
    images_per_class_train: int = 400
    images_per_class_test: int = 100
    captions_per_class: int = 50

    # Federated-style simulation
    num_clients: int = 6

    run_synthesis: bool = True
    synth_images_per_class: int = 150
    diffusion_steps: int = 25
    guidance_scale: float = 7.0

    batch_size: int = 32
    epochs: int = 50
    lr: float = 1e-4

    blip_model: str = "Salesforce/blip-image-captioning-base"
    clip_model: str = "openai/clip-vit-base-patch32"
    sd_model: str = "runwayml/stable-diffusion-v1-5"
    backbone: str = "resnet18"

    work_dir: str = "/kaggle/working/oscar_baseline_nico"

CFG = Config()

WORK = Path(CFG.work_dir)
WORK.mkdir(parents=True, exist_ok=True)
for sub in ["captions", "embeddings", "synthetic", "checkpoints", "metrics"]:
    (WORK / sub).mkdir(parents=True, exist_ok=True)

print(CFG)
print("Work dir:", WORK)

Config(dataset_name='nico_unique', data_root='/kaggle/input/datasets/khizaryy/nicoplusplus/NICO_unique', max_classes=30, images_per_class_train=400, images_per_class_test=100, captions_per_class=50, num_clients=6, run_synthesis=True, synth_images_per_class=150, diffusion_steps=25, guidance_scale=7.0, batch_size=32, epochs=50, lr=0.0001, blip_model='Salesforce/blip-image-captioning-base', clip_model='openai/clip-vit-base-patch32', sd_model='runwayml/stable-diffusion-v1-5', backbone='resnet18', work_dir='/kaggle/working/oscar_baseline_nico')
Work dir: /kaggle/working/oscar_baseline_nico


## 3) Dataset utilities



We pick the **top 30 classes by image count** to keep the baseline stable.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def is_image_file(p: Path) -> bool:
    return p.suffix.lower() in IMG_EXTS

def load_image(path: Path) -> Image.Image:
    return Image.open(path).convert("RGB")

def sample_list(items, k, rng):
    items = list(items)
    if k is None or k >= len(items):
        return items
    return rng.sample(items, k)

def count_class_images(root: Path):
    counts = []
    for d in sorted([x for x in root.iterdir() if x.is_dir()]):
        n = sum(1 for p in d.rglob("*") if p.is_file() and is_image_file(p))
        if n > 0:
            counts.append((d.name, n))
    counts.sort(key=lambda x: (-x[1], x[0]))
    return counts

def select_top_classes(root: Path, max_classes: int):
    counts = count_class_images(root)
    selected = [cls for cls, _ in counts[:max_classes]]
    return selected, counts

def build_imagefolder_records(root: Path, selected_classes, images_per_class_train: int, images_per_class_test: int, seed: int = 42):
    rng = random.Random(seed)
    train_records, test_records = [], []

    for cls in selected_classes:
        cls_dir = root / cls
        paths = [p for p in cls_dir.rglob("*") if p.is_file() and is_image_file(p)]
        paths = sorted(paths)
        if len(paths) == 0:
            continue

        rng.shuffle(paths)

        if len(paths) < 5:
            cut = max(1, len(paths) // 2)
        else:
            cut = max(1, int(0.8 * len(paths)))

        train_paths = paths[:cut]
        test_paths = paths[cut:]

        train_paths = sample_list(train_paths, images_per_class_train, rng)
        test_paths = sample_list(test_paths, images_per_class_test, rng)

        train_records.extend([(p, cls, "train") for p in train_paths])
        test_records.extend([(p, cls, "test") for p in test_paths])

    return train_records, test_records

class LabeledImageDataset(Dataset):
    def __init__(self, records, class_to_idx, transform=None):
        self.records = records
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        path, cls, _split = self.records[idx]
        image = load_image(Path(path))
        if self.transform is not None:
            image = self.transform(image)
        label = self.class_to_idx[cls]
        return image, label

def split_classes_across_clients(classes, num_clients):
    clients = [[] for _ in range(num_clients)]
    for i, cls in enumerate(classes):
        clients[i % num_clients].append(cls)
    return clients

def group_records_by_class(records):
    grouped = defaultdict(list)
    for p, cls, split in records:
        grouped[cls].append((p, split))
    return grouped

root = Path(CFG.data_root)
selected_classes, class_counts = select_top_classes(root, CFG.max_classes)
train_records, test_records = build_imagefolder_records(
    root,
    selected_classes=selected_classes,
    images_per_class_train=CFG.images_per_class_train,
    images_per_class_test=CFG.images_per_class_test,
    seed=SEED,
)

class_to_idx = {cls: i for i, cls in enumerate(selected_classes)}
train_grouped = group_records_by_class(train_records)
test_grouped = group_records_by_class(test_records)

print("Selected classes:", selected_classes)
print("Top class counts:", class_counts[:5])
print("Train records:", len(train_records))
print("Test records:", len(test_records))

Selected classes: ['dog', 'bear', 'chair', 'hat', 'train', 'kangaroo', 'flower', 'motorcycle', 'shrimp', 'scooter', 'cat', 'lizard', 'sunflower', 'tent', 'ostrich', 'squirrel', 'airplane', 'gun', 'bicycle', 'corn', 'truck', 'giraffe', 'butterfly', 'fox', 'mailbox', 'sailboat', 'dolphin', 'clock', 'bird', 'lion']
Top class counts: [('dog', 796), ('bear', 713), ('chair', 695), ('hat', 679), ('train', 665)]
Train records: 11862
Test records: 2968


## 4) Client simulation

OSCAR is one-shot federated learning, so we keep a small client split for the report and for later ablations.

In [5]:
client_classes = split_classes_across_clients(selected_classes, CFG.num_clients)
for i, cls_list in enumerate(client_classes):
    print(f"Client {i}: {cls_list}")

Client 0: ['dog', 'flower', 'sunflower', 'bicycle', 'mailbox']
Client 1: ['bear', 'motorcycle', 'tent', 'corn', 'sailboat']
Client 2: ['chair', 'shrimp', 'ostrich', 'truck', 'dolphin']
Client 3: ['hat', 'scooter', 'squirrel', 'giraffe', 'clock']
Client 4: ['train', 'cat', 'airplane', 'butterfly', 'bird']
Client 5: ['kangaroo', 'lizard', 'gun', 'fox', 'lion']


## 5) BLIP captioning

We caption only a small number of images per class while still using the OSCAR-style frozen vision-language step.

In [6]:
from transformers import BlipProcessor, BlipForConditionalGeneration

print("Loading BLIP...")
blip_processor = BlipProcessor.from_pretrained(CFG.blip_model)
blip_model = BlipForConditionalGeneration.from_pretrained(
    CFG.blip_model,
    torch_dtype=DTYPE if DEVICE == "cuda" else torch.float32,
).to(DEVICE)
blip_model.eval()

def caption_image(image_path: Path, max_new_tokens: int = 20):
    image = load_image(image_path)
    inputs = blip_processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = blip_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return blip_processor.decode(out[0], skip_special_tokens=True).strip()

captions_by_class = {}
caption_path = WORK / "captions" / f"{CFG.dataset_name}_captions.json"

for cls in tqdm(selected_classes, desc="Captioning classes"):
    cls_paths = [p for p, _split in train_grouped.get(cls, [])]
    cls_paths = sorted(cls_paths)[:CFG.captions_per_class]
    entries = []
    for p in tqdm(cls_paths, desc=f"{cls}", leave=False):
        try:
            cap = caption_image(p)
        except Exception:
            cap = f"a photo of {cls}"
        entries.append({"image": str(p), "caption": cap})
    captions_by_class[cls] = entries

with open(caption_path, "w") as f:
    json.dump(captions_by_class, f, indent=2)

print("Saved captions to:", caption_path)

Loading BLIP...


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

...

dolphin:   0%|          | 0/50 [00:00<?, ?it/s]

clock:   0%|          | 0/50 [00:00<?, ?it/s]

bird:   0%|          | 0/50 [00:00<?, ?it/s]

lion:   0%|          | 0/50 [00:00<?, ?it/s]

Saved captions to: /kaggle/working/oscar_baseline_nico/captions/nico_unique_captions.json


## 6) CLIP text embeddings



In [ ]:
from transformers import CLIPTextModel, CLIPTokenizer
import torch
import numpy as np
import json
import shutil
from tqdm.auto import tqdm

print("Loading SD-compatible CLIP model (768d)...")
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14").to(DEVICE)
text_encoder.eval()


embeddings_by_class = {}
emb_dir = WORK / "embeddings" / CFG.dataset_name
emb_dir.mkdir(parents=True, exist_ok=True)

for cls in tqdm(selected_classes, desc="Generating embeddings"):
    texts = [x["caption"] for x in captions_by_class.get(cls, []) if x["caption"]]
    if not texts:
        texts = [f"a photo of {cls}"]
    
    inputs = tokenizer(
        texts, 
        padding="max_length", 
        max_length=tokenizer.model_max_length, 
        truncation=True, 
        return_tensors="pt"
    ).to(DEVICE)
    
    with torch.no_grad():
        hidden_states = text_encoder(inputs.input_ids)[0]
    
    avg_emb = hidden_states.mean(dim=0, keepdim=True)

    avg_emb_np = avg_emb.detach().cpu().numpy()
    
    embeddings_by_class[cls] = {
        "captions": texts,
        "embedding": avg_emb_np.tolist(),
    }
    np.save(emb_dir / f"{cls}.npy", avg_emb_np)

emb_path = WORK / "embeddings" / f"{CFG.dataset_name}_clip_embeddings.json"
with open(emb_path, "w") as f:
    json.dump(embeddings_by_class, f, indent=2)

print("Saved embeddings to:", emb_path)
print("Embedding shape:", np.array(next(iter(embeddings_by_class.values()))["embedding"]).shape)

del text_encoder
del tokenizer
torch.cuda.empty_cache()

Loading SD-compatible CLIP model (768d)...


tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...23}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}

Generating embeddings:   0%|          | 0/30 [00:00<?, ?it/s]

Saved embeddings to: /kaggle/working/oscar_baseline_nico/embeddings/nico_unique_clip_embeddings.json
Embedding shape: (1, 77, 768)


## 7) Diffusion synthesis 
Creating synthetic data to preserve client data's privacy

In [ ]:
RUN_SYNTHESIS = CFG.run_synthesis

synthetic_root = WORK / "synthetic" / CFG.dataset_name
synthetic_exists = False

if RUN_SYNTHESIS:
    from diffusers import StableDiffusionPipeline

    print("Loading Stable Diffusion...")
    pipe = StableDiffusionPipeline.from_pretrained(
        CFG.sd_model,
        torch_dtype=DTYPE if DEVICE == "cuda" else torch.float32,
        safety_checker=None,
        requires_safety_checker=False,
    ).to(DEVICE)

    pipe.enable_attention_slicing()
    try:
        pipe.enable_vae_slicing()
    except Exception:
        pass

    if synthetic_root.exists():
        shutil.rmtree(synthetic_root)
    synthetic_root.mkdir(parents=True, exist_ok=True)

    for cls in tqdm(selected_classes, desc="Generating synthetic images"):
        class_dir = synthetic_root / cls
        class_dir.mkdir(parents=True, exist_ok=True)
        
        emb_list = embeddings_by_class[cls]["embedding"]
        
        class_embedding = torch.tensor(
            emb_list, 
            dtype=DTYPE if DEVICE == "cuda" else torch.float32
        ).to(DEVICE)

        for j in range(CFG.synth_images_per_class):
            gen = torch.Generator(device=DEVICE).manual_seed(SEED + 1000 + j)
            try:
                if DEVICE == "cuda":
                    with torch.autocast(device_type="cuda", dtype=torch.float16):
                        out = pipe(
                            prompt_embeds=class_embedding, 
                            num_inference_steps=CFG.diffusion_steps,
                            guidance_scale=CFG.guidance_scale,
                            generator=gen,
                        )
                else:
                    out = pipe(
                        prompt_embeds=class_embedding, 
                        num_inference_steps=CFG.diffusion_steps,
                        guidance_scale=CFG.guidance_scale,
                        generator=gen,
                    )
                img = out.images[0]
                img.save(class_dir / f"{cls}_{j:03d}.png")
            except Exception as e:
                print(f"[warn] failed on {cls}#{j}: {e}")

    synthetic_exists = synthetic_root.exists() and any(synthetic_root.iterdir())
    print("Synthetic dataset created:", synthetic_exists)
else:
    print("Synthesis disabled. Training will use the sampled real images as fallback.")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading Stable Diffusion...


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

...

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

Synthetic dataset created: True


## 8) Train ResNet-18

Using resnet 18 for faster computation for slight loss in accuracy, ensures we can easily train on kaggle in reasonable timeframe


In [9]:
class LabeledRecordsDataset(Dataset):
    def __init__(self, records, class_to_idx, transform=None):
        self.records = records
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        path, cls, _split = self.records[idx]
        image = load_image(Path(path))
        if self.transform is not None:
            image = self.transform(image)
        label = self.class_to_idx[cls]
        return image, label

train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

if synthetic_exists:
    train_ds = torchvision.datasets.ImageFolder(str(synthetic_root), transform=train_transform)
    class_to_idx_used = train_ds.class_to_idx
    train_source = "synthetic"
else:
    class_to_idx_used = class_to_idx
    train_ds = LabeledRecordsDataset(train_records, class_to_idx_used, transform=train_transform)
    train_source = "real-sampled"

test_ds = LabeledRecordsDataset(test_records, class_to_idx_used, transform=test_transform)

print("Training source:", train_source)
print("Classes used:", class_to_idx_used)
print("Train size:", len(train_ds))
print("Test size:", len(test_ds))

Training source: synthetic
Classes used: {'airplane': 0, 'bear': 1, 'bicycle': 2, 'bird': 3, 'butterfly': 4, 'cat': 5, 'chair': 6, 'clock': 7, 'corn': 8, 'dog': 9, 'dolphin': 10, 'flower': 11, 'fox': 12, 'giraffe': 13, 'gun': 14, 'hat': 15, 'kangaroo': 16, 'lion': 17, 'lizard': 18, 'mailbox': 19, 'motorcycle': 20, 'ostrich': 21, 'sailboat': 22, 'scooter': 23, 'shrimp': 24, 'squirrel': 25, 'sunflower': 26, 'tent': 27, 'train': 28, 'truck': 29}
Train size: 4500
Test size: 2968


In [10]:
train_loader = DataLoader(
    train_ds,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE == "cuda"),
)

test_loader = DataLoader(
    test_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE == "cuda"),
)

model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(class_to_idx_used))
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr)

best_acc = 0.0
best_path = WORK / "checkpoints" / f"{CFG.dataset_name}_resnet18_best.pt"
history = []

def run_eval(model, loader):
    model.eval()
    total, correct = 0, 0
    losses = []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = criterion(logits, y)
            pred = logits.argmax(dim=1)
            total += y.size(0)
            correct += (pred == y).sum().item()
            losses.append(loss.item())
    return (float(np.mean(losses)) if losses else 0.0), (correct / max(total, 1))

for epoch in range(CFG.epochs):
    model.train()
    running = []
    for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{CFG.epochs}"):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        running.append(loss.item())

    train_loss = float(np.mean(running)) if running else 0.0
    val_loss, val_acc = run_eval(model, test_loader)
    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "test_loss": val_loss,
        "test_acc": val_acc,
    })
    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}  test_loss={val_loss:.4f}  test_acc={val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "class_to_idx": class_to_idx_used,
            "config": CFG.__dict__,
        }, best_path)

print("Best test accuracy:", best_acc)
print("Checkpoint:", best_path)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth



  0%|          | 0.00/44.7M [00:00<?, ?B/s]
 16%|█▌        | 7.12M/44.7M [00:00<00:00, 74.6MB/s]
 51%|█████     | 22.6M/44.7M [00:00<00:00, 126MB/s] 
100%|██████████| 44.7M/44.7M [00:00<00:00, 140MB/s]


Epoch 1/50:   0%|          | 0/141 [00:00<?, ?it/s]

Epoch 1: train_loss=0.5942  test_loss=1.7593  test_acc=0.5468


Epoch 2/50:   0%|          | 0/141 [00:00<?, ?it/s]

...

Epoch 48: train_loss=0.0002  test_loss=2.5061  test_acc=0.5145


Epoch 49/50:   0%|          | 0/141 [00:00<?, ?it/s]

Epoch 49: train_loss=0.0002  test_loss=2.4941  test_acc=0.5202


Epoch 50/50:   0%|          | 0/141 [00:00<?, ?it/s]

Epoch 50: train_loss=0.0003  test_loss=2.5080  test_acc=0.5152
Best test accuracy: 0.5579514824797843
Checkpoint: /kaggle/working/oscar_baseline_nico/checkpoints/nico_unique_resnet18_best.pt


## 9) Save metrics

This cell writes the config, class list, captions, embeddings, and training history so your deliverable stays reproducible.

In [11]:
metrics = {
    "dataset": CFG.dataset_name,
    "selected_classes": selected_classes,
    "class_counts_top": class_counts[:CFG.max_classes],
    "num_clients": CFG.num_clients,
    "train_records": len(train_records),
    "test_records": len(test_records),
    "synthetic_exists": synthetic_exists,
    "train_source": train_source,
    "best_test_accuracy": best_acc,
    "history": history,
}

metrics_path = WORK / "metrics" / f"{CFG.dataset_name}_summary.json"
history_path = WORK / "metrics" / f"{CFG.dataset_name}_history.csv"

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

pd.DataFrame(history).to_csv(history_path, index=False)

print("Saved metrics to:", metrics_path)
print("Saved history to:", history_path)

Saved metrics to: /kaggle/working/oscar_baseline_nico/metrics/nico_unique_summary.json
Saved history to: /kaggle/working/oscar_baseline_nico/metrics/nico_unique_history.csv


## Analysis:

Original Oscar paper had an accuracy in 70s for this dataset, our current model is computationally inexpensive compared to oscar paper and only follows the same pipeline, which comes at this loss of accuracy. This model allows future ablations to be easily performed and to see if accuracy is preserved in the top k vector compression. Current model overfits the data and seems to memorize the dataset due to lack of regularization, and scheduled learning rate. The reduced accuracy is also linked to fewer steps used in diffusion compared to actual oscar model. We will refine the baseline once more before our final deliverable, this could not be completed within this time due to the extensive training time hindering us from fine tuning parameters for a model with balance between accuracy and computation. 